# CubbyLLM 0068: route 5 on an A100 (grilly2)

The home card's three timings, run once on Colab's A100 with grilly2 built here from source:
1. **the head alone** (`head_lab_time.py --route card`): route 5 against a full 32k head and the full 128k head;
2. **inside a real decode step** (`head_lab_decode.py`): at 128k and on the synthetic 256k head;
3. **through the wired `generate`** (`head_lab_generate.py`): the seats' own call.

About 25 minutes: Rust and the Vulkan SDK (2 min), grilly2's build (5 to 10 min), the export (2 min), the three runs
(about 10 min). Nothing is trained and no weight changes. The logs land in `MyDrive/cubbyllm/head_shortlist/a100/`;
Cowork reads them at home and commits them. **Use an A100 runtime, run all, then disconnect the runtime.**

grilly2 has not run on Colab before (grilly v1 has); cell 3 stops with a clear message if Vulkan does not see the card.

In [ ]:
# --- 1. the code: CubbyLLM and grilly2 at the commits the order names ---
import os, sys, glob, json, shutil, hashlib, zipfile, datetime, subprocess
from google.colab import drive; drive.mount('/content/drive')
CUBBY_BRANCH = 'cowork/0068'
G2_BRANCH, G2_COMMIT = 'cowork/shortlist-generate', '7cb24f8'

def run(args, cwd=None, env=None):
    print('$', ' '.join(args), flush=True)
    p = subprocess.Popen(args, cwd=cwd, env=env, text=True, bufsize=1, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end='')
    assert p.wait() == 0, f'exit {p.returncode}: {" ".join(args[:3])}'

if not os.path.exists('/content/CubbyLLM'):
    run(['git', 'clone', '-q', '-b', CUBBY_BRANCH, '--single-branch', 'https://github.com/Grillcheese-AI/CubbyLLM.git',
         '/content/CubbyLLM'])
else:
    run(['git', 'pull', '-q', '--ff-only'], cwd='/content/CubbyLLM')
if not os.path.exists('/content/grilly2'):
    run(['git', 'clone', '-q', '--recursive', '-b', G2_BRANCH, 'https://github.com/Grillcheese-AI/grilly2.git',
         '/content/grilly2'])
run(['git', 'checkout', '-q', G2_COMMIT], cwd='/content/grilly2')
run(['git', 'log', '--oneline', '-1'], cwd='/content/CubbyLLM')
run(['git', 'log', '--oneline', '-1'], cwd='/content/grilly2')
run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'])

In [ ]:
# --- 2. build grilly2: the LunarG SDK (headers, glslc, shaderc), Rust, maturin ---
# The SDK as LunarG's Linux getting-started page sets it up: the tarball unpacked, then setup-env.sh's variables
# (VULKAN_SDK, PATH, LD_LIBRARY_PATH; newer SDKs keep the loader in lib/VulkanLoader/lib). If the download fails,
# take the Linux tarball from https://vulkan.lunarg.com/sdk/home, put it in MyDrive/cubbyllm/vulkansdk/ and run
# this cell again.
SDK_VER = '1.4.363.1'                                   # the SDK that worked on Colab (2026-11-18)
ROOT = '/content/vulkansdk'
TAR = f'vulkansdk-linux-x86_64-{SDK_VER}.tar.xz'
if not os.path.exists(f'{ROOT}/{SDK_VER}/x86_64/bin/glslc'):
    os.makedirs(ROOT, exist_ok=True)
    kept = f'/content/drive/MyDrive/cubbyllm/vulkansdk/{TAR}'
    if os.path.exists(kept):
        shutil.copy(kept, f'/content/{TAR}')
    else:
        got = subprocess.run(['curl', '-fsSL', '-o', f'/content/{TAR}',
                              f'https://sdk.lunarg.com/sdk/download/{SDK_VER}/linux/{TAR}'])
        assert got.returncode == 0, f'the download failed: put {TAR} in MyDrive/cubbyllm/vulkansdk/ and rerun'
    run(['tar', 'xf', f'/content/{TAR}', '-C', ROOT])
SDK = f'{ROOT}/{SDK_VER}/x86_64'
LOADER = next((d for d in (f'{SDK}/lib/VulkanLoader/lib', f'{SDK}/lib') if glob.glob(f'{d}/libvulkan.so*')), None)
assert os.path.exists(f'{SDK}/bin/glslc') and os.path.exists(f'{SDK}/lib/libshaderc_combined.a') and LOADER, \
    f'the SDK is not laid out as expected: look in {ROOT}/{SDK_VER}'
CARGO = os.path.expanduser('~/.cargo/bin')
if not os.path.exists(f'{CARGO}/cargo'):
    run(['bash', '-c', 'curl --proto =https --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'maturin'])
BUILD = dict(os.environ, VULKAN_SDK=SDK, PATH=f'{CARGO}:{SDK}/bin:' + os.environ['PATH'],
             LD_LIBRARY_PATH=f'{LOADER}:{SDK}/lib:' + os.environ.get('LD_LIBRARY_PATH', ''),
             RUSTFLAGS=f'-L native={LOADER}')          # grilly2 links -lvulkan from VULKAN_SDK/lib only
shutil.rmtree('/content/g2dist', ignore_errors=True)
run([sys.executable, '-m', 'maturin', 'build', '--release', '--out', '/content/g2dist', '--auditwheel', 'skip'],
    cwd='/content/grilly2', env=BUILD)
wheel = glob.glob('/content/g2dist/*.whl')[0]
with zipfile.ZipFile(wheel) as z:                       # the extension beside the checkout's python/, as at home,
    for m in z.namelist():                              # so each log names grilly2's commit
        if m.startswith('grilly/_C'):
            open(f'/content/grilly2/python/{m}', 'wb').write(z.read(m))
print('built', os.path.basename(wheel))

In [ ]:
# --- 3. Vulkan sees the A100? grilly2 refuses a CPU driver, so it needs NVIDIA's own Vulkan driver ---
# Colab mounts the driver's CUDA libraries, not always its Vulkan part (libGLX_nvidia and the libraries beside it).
# When grilly2 does not see the card, this takes that part from NVIDIA's installer for the exact driver version the
# card runs (a few hundred MB from NVIDIA's site): extracted only, no kernel module, nothing installed system-wide.
def doctor(env):
    d = subprocess.run([sys.executable, '-m', 'grilly.doctor'], env=env, text=True, capture_output=True)
    lines = d.stdout.splitlines()
    dev = next((x for x in lines if x.startswith('[ok]') and ' device ' in x), '')
    return d, 'NVIDIA' in dev and any(x.startswith('[ok]') and 'a kernel ran' in x for x in lines)

VER = subprocess.run(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], text=True,
                     capture_output=True).stdout.split()[0]
NV = [d for d in ('/usr/lib64-nvidia', '/usr/lib/x86_64-linux-gnu') if os.path.isdir(d)]
ENV = dict(os.environ, PYTHONPATH='/content/grilly2/python', PYTHONIOENCODING='utf-8',
           LD_LIBRARY_PATH=':'.join(NV + [LOADER, f'{SDK}/lib', os.environ.get('LD_LIBRARY_PATH', '')]))
doc, ok = doctor(ENV)
print('driver', VER, '| Vulkan parts the runtime mounts:',
      sorted(os.path.basename(p) for d in NV for p in glob.glob(f'{d}/libGLX_nvidia*') + glob.glob(f'{d}/libnvidia-gl*')),
      '| ICD files:', glob.glob('/usr/share/vulkan/icd.d/*.json') + glob.glob('/etc/vulkan/icd.d/*.json'))
if not ok:
    RUN = f'NVIDIA-Linux-x86_64-{VER}.run'
    if not os.path.exists(f'/content/{RUN}'):
        for url in (f'https://us.download.nvidia.com/tesla/{VER}/{RUN}',
                    f'https://download.nvidia.com/XFree86/Linux-x86_64/{VER}/{RUN}'):
            if subprocess.run(['curl', '-fsSL', '-o', f'/content/{RUN}', url]).returncode == 0:
                break
        else:
            raise AssertionError(f"NVIDIA's installer for driver {VER} was not found: send Cowork this message")
    if not os.path.isdir(f'/content/NVIDIA-Linux-x86_64-{VER}'):
        run(['sh', f'/content/{RUN}', '--extract-only'], cwd='/content')
    VK = '/content/nvidia_vulkan'
    os.makedirs(VK, exist_ok=True)
    for pat in ('libGLX_nvidia.so.*', 'libnvidia-glcore.so.*', 'libnvidia-glvkspirv.so.*', 'libnvidia-gpucomp.so.*',
                'libnvidia-tls.so.*', 'libGLdispatch.so.*'):
        for p in glob.glob(f'/content/NVIDIA-Linux-x86_64-{VER}/{pat}'):
            shutil.copy(p, VK)
    icd = f'{VK}/nvidia_icd.json'
    json.dump({'file_format_version': '1.0.0',
               'ICD': {'library_path': glob.glob(f'{VK}/libGLX_nvidia.so.*')[0], 'api_version': '1.3'}}, open(icd, 'w'))
    run(['bash', '-c', 'apt-get -qq update && apt-get -qq install -y libx11-6 libxext6 > /dev/null'])
    ENV.update(LD_LIBRARY_PATH=f'{VK}:' + ENV['LD_LIBRARY_PATH'], VK_DRIVER_FILES=icd, VK_ICD_FILENAMES=icd)
    doc, ok = doctor(ENV)
subprocess.run([f'{SDK}/bin/vulkaninfo', '--summary'], env=ENV)
print(doc.stdout, doc.stderr)
assert ok, 'grilly2 does not see the A100 through Vulkan. Send Cowork the output of this cell.'

In [ ]:
# --- 4. the inputs: the base exported from the checkpoint 0068 read, the route's index, the 50 timed states ---
DRIVE = '/content/drive/MyDrive/cubbyllm'
CK450 = '/content/base450m_cont_final.pt'
if not os.path.exists(CK450):                           # Drive-FUSE reads are slow: one copy to the local disk
    shutil.copy(f'{DRIVE}/runs/base450m_cont/base450m_cont_final.pt', CK450)
EXPORT = '/content/export'
if not os.path.exists(f'{EXPORT}/model.safetensors'):
    run([sys.executable, 'validation/export_base.py', '--ckpt', CK450, '--out', EXPORT], cwd='/content/CubbyLLM')
from safetensors import safe_open
import numpy as np
with safe_open(f'{EXPORT}/model.safetensors', framework='numpy') as f:
    head = np.ascontiguousarray(f.get_tensor('lm_head.weight'), dtype=np.float32)
HEAD_SHA = '63fcdfe23e7e16617443d2c4b26db743bb89146d56bbb7d03f3c755584580cd1'   # the home export's head rows
assert head.shape == (128000, 1024) and hashlib.sha256(head.tobytes()).hexdigest() == HEAD_SHA, \
    "the exported head is not the one the route's index was built on"
del head
DUMPS = '/content/head_shortlist'
os.makedirs(DUMPS, exist_ok=True)
for name in ('index_router_succ.npz', 'heldout_b.npz'):
    if not os.path.exists(f'{DUMPS}/{name}'):
        shutil.copy(f'{DRIVE}/head_shortlist/{name}', f'{DUMPS}/{name}')
SHA = {'index_router_succ.npz': '814c8e03245161591d697724a1931e01549e22e52d06012ed4b99d6913fbdd15',
       'heldout_b.npz': 'eef2d754ba806991707ec4deff9edeeced277b72a51e3bcfbc42ca3df76de448'}
for name, want in SHA.items():
    assert hashlib.sha256(open(f'{DUMPS}/{name}', 'rb').read()).hexdigest() == want, f'{name} is not the one recorded'
CACHE = f'{DRIVE}/token_cache_cont'
print('inputs checked')

In [ ]:
# --- 5. the three timings (about 10 min), as at home ---
common = ['--export', EXPORT, '--dumps', DUMPS]
run([sys.executable, '-u', 'validation/head_lab_time.py', '--route', 'card', '--dump', 'heldout_b'] + common,
    cwd='/content/CubbyLLM', env=ENV)
run([sys.executable, '-u', 'validation/head_lab_decode.py', '--cache', CACHE] + common, cwd='/content/CubbyLLM', env=ENV)
run([sys.executable, '-u', 'validation/head_lab_generate.py', '--cache', CACHE] + common, cwd='/content/CubbyLLM',
    env=ENV)

In [ ]:
# --- 6. keep the logs on Drive (named _a100), with the card and driver they ran on ---
OUT = f'{DRIVE}/head_shortlist/a100'
os.makedirs(OUT, exist_ok=True)
today = datetime.date.today().isoformat()
logs = [p for k in ('head_lab_time_grilly_card', 'head_lab_decode', 'head_lab_generate')
        for p in glob.glob(f'/content/CubbyLLM/validation/logs/{k}_{today}_{G2_COMMIT}.json')]
assert len(logs) == 3, logs
for p in logs:
    shutil.copy(p, f'{OUT}/' + os.path.basename(p).replace('.json', '_a100.json'))
smi = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], text=True,
                     capture_output=True).stdout
open(f'{OUT}/a100_env_{today}.txt', 'w').write(smi + '\n' + doc.stdout)
print('kept on Drive:', sorted(os.listdir(OUT)))
print('Done: disconnect the runtime.')